# AllianceAI — Interactive Demo

This notebook drives the full analysis pipeline from inside the container.
The DuckDB learning store lives on a persistent volume, so every run keeps
improving the calibration across container restarts.

**Sections**
1. Run a full analysis on a ticker
2. Investment stance + confidence
3. Highlights & opportunity outlook
4. Real data vs past predictions (the learning loop)
5. Walk-forward backtest (train the AI from history)
6. View the interactive HTML report inline

In [ ]:
from allianceai.orchestrator import analyse

TICKER = "AAPL"   # try MSFT, NVDA, MELI, NOW ...
result = analyse(TICKER, output_dir="/app/reports")
print("done:", result["ticker"])

## 2. Investment stance + confidence

In [ ]:
d = result["decision"]
print(f"VERDICT: {d['verdict']}  ({d['score']}/100)")
print(f"Confidence: {d['confidence']['level']} ({d['confidence']['score']}/100)")
print(f"\n{d['summary']}\n{d['timing']}\n")
for r in d["rationale"]:
    print(" -", r)

dc = result.get("data_confidence")
if dc:
    print(f"\nAnalysis data confidence: {dc['level']} ({dc['score']}/100)")

## 3. Highlights & opportunity outlook

In [ ]:
print("HIGHLIGHTS")
for h in result.get("highlights", []):
    print("  -", h)
print("\nOPPORTUNITY OUTLOOK")
for o in result.get("opportunities", []):
    print("  +", o)

## 4. Real data vs past predictions (the learning loop)

In [ ]:
acc = result.get("prediction_accuracy")
acc if acc is not None and not acc.empty else "No graded predictions yet — run a backtest below to seed them."

## 5. Walk-forward backtest — train the AI from history

Modular: deep history for maximum samples, or recent windows for the cleanest
calibration. Defaults to the fast Holt-Winters path (seconds, no API calls).

In [ ]:
from allianceai.data.fetcher import fetch_all
from allianceai.models.backtest import backtest_ticker, backtest_summary

data = fetch_all(TICKER)
graded = backtest_ticker(
    TICKER, data["income"], data["balance"], data["cashflow"],
    max_steps=12,   # recent windows = cleanest calibration; drop for full history
)
backtest_summary(graded)

## 6. View the interactive HTML report inline

In [ ]:
from IPython.display import IFrame
IFrame(src=f"/files/reports/{TICKER}_report.html", width="100%", height=900)